# BadBlueprint full scoring (Colab)

This notebook runs the BadBlueprint **full scoring** flow using:
- **Model constraint**: `openai/gpt-oss-20b` (open-weight; no proprietary API advantage)
- A **local OpenAI-compatible endpoint** started inside Colab via `transformers serve`
- The **upstream** `agentbeats-lambda` harness (forked) cloned into `/content/agentbeats-lambda`

Outputs are normalized to `results/badblueprint/*` and packaged for download.


## A. Runtime / GPU check


In [ ]:
import platform
import shutil
import subprocess

print("Python:", platform.python_version())

def run(cmd: str) -> int:
    print(f"$ {cmd}")
    p = subprocess.run(cmd, shell=True, check=False)
    return p.returncode

# GPU info (best-effort)
run("nvidia-smi || true")

# Torch CUDA check (Colab usually has torch preinstalled)
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("WARNING: torch check failed:", repr(e))

# System memory / disk
run("free -h || true")
run("df -h / || true")

if shutil.which("nvidia-smi") is None:
    print("WARNING: No GPU detected. Full scoring may be extremely slow or fail.")



## B. Environment setup


In [ ]:
import subprocess

def sh(cmd: str):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# Uninstall potentially conflicting packages (safe no-op if missing)
for pkg in ["torchvision", "torchaudio"]:
    sh(f"pip uninstall -y {pkg} || true")

# Install Transformers with serving extras (recommended by HF docs)
# Avoid installing from git main for reproducibility.
sh('pip install -U --quiet "transformers[serving]" "huggingface-hub>=0.34.0,<1.0"')

# Core runtime deps
sh("pip install -U --quiet accelerate safetensors requests")

# Optional: faster HF transfers when supported
sh("pip install -U --quiet hf_transfer || true")



In [ ]:
sh('python -c "import transformers, huggingface_hub; print(transformers.__version__, huggingface_hub.__version__)"')


## C. Model download


In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import os

MODEL_ID = "openai/gpt-oss-20b"
LOCAL_DIR = Path("/content/models/gpt-oss-20b")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# Optional: set HF_TOKEN in Colab Secrets or environment (do not print it)
hf_token = os.environ.get("HF_TOKEN")

print("Downloading model (may take a while)...")
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(LOCAL_DIR),
    token=hf_token,
)

file_count = sum(1 for _ in LOCAL_DIR.rglob("*"))
print(f"Model downloaded to: {LOCAL_DIR} (files: {file_count})")



## D. Start local OpenAI-compatible endpoint


In [ ]:
import os
import signal
import subprocess
import time
from pathlib import Path

MODEL_ID = "openai/gpt-oss-20b"
ENDPOINT = "http://127.0.0.1:8000/v1"
SERVER_LOG = Path("/content/transformers_server.log")
SERVER_PID = Path("/content/transformers_server.pid")

def is_pid_running(p: int) -> bool:
    try:
        os.kill(p, 0)
        return True
    except ProcessLookupError:
        return False

def stop_server():
    if not SERVER_PID.exists():
        return
    pid = int(SERVER_PID.read_text().strip())
    try:
        os.killpg(pid, signal.SIGTERM)
        time.sleep(2)
        if is_pid_running(pid):
            os.killpg(pid, signal.SIGKILL)
        print(f"Stopped server process group {pid}")
    except ProcessLookupError:
        print("Server process not running.")
    finally:
        SERVER_PID.unlink(missing_ok=True)

def start_server():
    # Remove stale PID
    if SERVER_PID.exists():
        pid = int(SERVER_PID.read_text().strip())
        if not is_pid_running(pid):
            print("Found stale PID file; removing.")
            SERVER_PID.unlink(missing_ok=True)

    if SERVER_PID.exists():
        print("Server appears to be running already. Skipping start.")
        return

    cmd = [
        "transformers",
        "serve",
        "--port",
        "8000",
        "--force-model",
        MODEL_ID,
    ]
    print("Starting server:", " ".join(cmd))
    with SERVER_LOG.open("w") as log_f:
        proc = subprocess.Popen(
            cmd,
            stdout=log_f,
            stderr=subprocess.STDOUT,
            preexec_fn=os.setsid,  # process group for killpg
        )
    SERVER_PID.write_text(str(proc.pid))
    time.sleep(5)
    print(f"Server PID: {proc.pid}")
    print(f"Server log: {SERVER_LOG}")

start_server()
print("Model endpoint:", ENDPOINT)



### Healthcheck


In [ ]:
import time
import requests

ENDPOINT = "http://127.0.0.1:8000/v1"

ok = False
last_err = None
for _ in range(20):
    try:
        resp = requests.get(f"{ENDPOINT}/models", timeout=5)
        if resp.status_code == 200:
            print("Server is healthy.")
            data = resp.json()
            print(data)
            ok = True
            break
        last_err = f"HTTP {resp.status_code}: {resp.text[:200]}"
    except Exception as exc:
        last_err = repr(exc)
    time.sleep(3)

if not ok:
    raise RuntimeError(f"Server healthcheck failed: {last_err}")



## E. Clone repo and prepare submission bundle


In [ ]:
import subprocess
from pathlib import Path

def run(cmd: str, cwd: Path | None = None):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=str(cwd) if cwd else None)

harness_dir = Path("/content/agentbeats-lambda")
if not harness_dir.exists():
    run("git clone https://github.com/purplevanguard/agentbeats-lambda.git", cwd=Path("/content"))

if (harness_dir / ".git").exists():
    run("git fetch --prune origin", cwd=harness_dir)
    run("git checkout -B submission/purple_vanguard origin/submission/purple_vanguard", cwd=harness_dir)
    run("git reset --hard origin/submission/purple_vanguard", cwd=harness_dir)

print("Harness dir:", harness_dir)


No manual copy needed. This notebook assumes the submission bundle is already present in the harness branch `submission/purple_vanguard` at:
`/content/agentbeats-lambda/scenarios/security_arena/submissions/purple_vanguard/badblueprint/`


In [ ]:
from pathlib import Path

bundle_dir = Path("/content/agentbeats-lambda/scenarios/security_arena/submissions/purple_vanguard/badblueprint")
toml_path = bundle_dir / "scenario_badblueprint.toml"
plugin_path = bundle_dir / "plugin.py"
artifacts_dir = bundle_dir / "artifacts"
prompts_dir = bundle_dir / "prompts"

missing = []
if not toml_path.exists():
    missing.append(str(toml_path))
if not plugin_path.exists():
    missing.append(str(plugin_path))
if not artifacts_dir.is_dir() or not any(artifacts_dir.rglob("*")):
    missing.append(str(artifacts_dir))
if not prompts_dir.is_dir() or not any(prompts_dir.rglob("*")):
    missing.append(str(prompts_dir))

if missing:
    raise RuntimeError(
        "Missing required files in the harness branch submission/purple_vanguard. Sync the submission bundle to agentbeats-lambda branch `submission/purple_vanguard`."
    )

print("Submission bundle looks OK at", bundle_dir)


In [ ]:
from pathlib import Path

harness_dir = Path("/content/agentbeats-lambda")
toml_path = harness_dir / "scenarios/security_arena/submissions/purple_vanguard/badblueprint/scenario_badblueprint.toml"

if not toml_path.exists():
    raise FileNotFoundError(f"Missing scenario toml: {toml_path}")

toml_text = toml_path.read_text(encoding="utf-8", errors="ignore")
bad_tokens = []
if "vendor/agentbeats-lambda/" in toml_text:
    bad_tokens.append("vendor/agentbeats-lambda/")
if "scripts/run_badblueprint_orchestrator.py" in toml_text:
    bad_tokens.append("scripts/run_badblueprint_orchestrator.py")
if bad_tokens:
    raise RuntimeError(
        "TOML still contains old paths: " + ", ".join(bad_tokens)
        + "\nFix scenario_badblueprint.toml in agentbeats-lambda branch `submission/purple_vanguard` to point to harness paths under `scenarios/security_arena/...`."
    )

must_exist = [
    harness_dir / "scenarios/security_arena/orchestrator.py",
    harness_dir / "scenarios/security_arena/agents/generic_attacker.py",
    harness_dir / "scenarios/security_arena/agents/generic_defender.py",
]
missing = [str(p) for p in must_exist if not p.exists()]
if missing:
    raise FileNotFoundError("Missing harness files:\n" + "\n".join(missing))
print("TOML + harness preflight guard passed.")


## F. Install harness


In [ ]:
import subprocess
import sys
from pathlib import Path

harness_dir = Path("/content/agentbeats-lambda")
if not harness_dir.exists():
    raise RuntimeError(f"Harness repo not found at {harness_dir}")

subprocess.run(f"pip install -e {harness_dir}", shell=True, check=True, cwd=harness_dir)

# Detect CLI from harness pyproject scripts
pyproject = harness_dir / "pyproject.toml"
cli_name = None
if pyproject.exists():
    import tomllib
    data = tomllib.loads(pyproject.read_text())
    scripts = data.get("project", {}).get("scripts", {})
    if scripts:
        cli_name = sorted(scripts.keys())[0]

if cli_name is None:
    # Fallback: find likely binaries in current python bin dir
    bin_dir = Path(sys.executable).parent
    candidates = [p.name for p in bin_dir.iterdir() if p.is_file() and ("agent" in p.name or "beats" in p.name)]
    cli_name = candidates[0] if candidates else None

if not cli_name:
    raise RuntimeError("Could not detect harness CLI entrypoint.")

CLI_NAME = cli_name  # keep as a global
print("Detected harness CLI:", CLI_NAME)


## G. Configure harness to use local endpoint


In [ ]:
import os
import re
from pathlib import Path

harness_dir = Path("/content/agentbeats-lambda")
if not harness_dir.exists():
    raise RuntimeError(f"Harness repo not found at {harness_dir}")

ENDPOINT = "http://127.0.0.1:8000/v1"

ENV_PATTERNS = [
    re.compile(r'os\.environ\[\s*"([A-Z0-9_]+)"\s*\]'),
    re.compile(r"os\.environ\[\s*'([A-Z0-9_]+)'\s*\]"),
    re.compile(r'getenv\(\s*"([A-Z0-9_]+)"\s*\)'),
    re.compile(r"getenv\(\s*'([A-Z0-9_]+)'\s*\)"),
]

def iter_text_files(root: Path):
    for p in root.rglob("*"):
        if p.is_file() and p.suffix in {".py", ".toml", ".yaml", ".yml", ".md"}:
            yield p

env_names = set()
for p in iter_text_files(harness_dir):
    try:
        txt = p.read_text(errors="ignore")
    except Exception:
        continue
    for pat in ENV_PATTERNS:
        for m in pat.findall(txt):
            env_names.add(m)

if not env_names:
    raise RuntimeError("No environment variables found in harness source. Cannot configure endpoint reliably.")

endpoint_vars = [
    n for n in env_names
    if any(k in n for k in ["BASE_URL", "API_BASE", "ENDPOINT", "HOST", "URL"])
    and "MODEL" not in n
    and "KEY" not in n
]
key_vars = [n for n in env_names if "API_KEY" in n or n.endswith("_KEY")]
model_vars = [
    n for n in env_names
    if "MODEL" in n and not any(k in n for k in ["ENDPOINT", "BASE", "URL", "HOST"])
]

for n in endpoint_vars:
    os.environ[n] = ENDPOINT

for n in key_vars:
    os.environ.setdefault(n, "DUMMY_KEY")  # never print real secrets

for n in model_vars:
    os.environ.setdefault(n, "gpt-oss-20b")

os.environ.setdefault("MODEL_NAME", "gpt-oss-20b")

print("Configured endpoint:", ENDPOINT)
print("Set endpoint vars:", ", ".join(sorted(endpoint_vars)) if endpoint_vars else "(none found)")
print("Set key vars:", ", ".join(sorted(key_vars)) if key_vars else "(none found)")
print("Set model vars:", ", ".join(sorted(model_vars)) if model_vars else "(none found)")
print("MODEL_NAME:", os.environ.get("MODEL_NAME"))


## H. Run FULL SCORING


In [ ]:
import shutil
import subprocess
from pathlib import Path

def _supports_score_subcmd(cli_base: list[str]) -> bool:
    try:
        p = subprocess.run(
            cli_base + ["score", "--help"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
            text=True,
        )
        return p.returncode == 0
    except Exception:
        return False

def run_full_scoring(show_logs: bool = True, mode: str = "full", timeout_sec: int | None = None):
    harness_dir = Path("/content/agentbeats-lambda")
    toml_path = harness_dir / "scenarios/security_arena/submissions/purple_vanguard/badblueprint/scenario_badblueprint.toml"
    results_dir = Path("/content/results/badblueprint")
    results_dir.mkdir(parents=True, exist_ok=True)
    log_path = results_dir / "full_score.log"

    if not toml_path.exists():
        raise FileNotFoundError(f"Missing scenario toml: {toml_path}")

    s = toml_path.read_text(encoding="utf-8", errors="ignore")
    bad_tokens = []
    if "vendor/agentbeats-lambda/" in s:
        bad_tokens.append("vendor/agentbeats-lambda/")
    if "scripts/run_badblueprint_orchestrator.py" in s:
        bad_tokens.append("scripts/run_badblueprint_orchestrator.py")
    if bad_tokens:
        raise RuntimeError(
            "TOML still contains old paths: " + ", ".join(bad_tokens)
            + "
Fix scenario_badblueprint.toml in agentbeats-lambda branch `submission/purple_vanguard` "
              "to point to harness paths under `scenarios/security_arena/...`."
        )

    must_exist = [
        harness_dir / "scenarios/security_arena/orchestrator.py",
        harness_dir / "scenarios/security_arena/agents/generic_attacker.py",
        harness_dir / "scenarios/security_arena/agents/generic_defender.py",
    ]
    missing = [str(p) for p in must_exist if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing harness files:
" + "
".join(missing))

    if shutil.which("uv"):
        cli_base = ["uv", "run", "agentbeats-run"]
    else:
        cli_base = ["agentbeats-run"]

    supports_score = _supports_score_subcmd(cli_base)

    if supports_score:
        cmd = cli_base + ["score", "--mode", mode, str(toml_path)]
        if show_logs:
            cmd += ["--show-logs"]
    else:
        cmd = cli_base + [str(toml_path)]
        if show_logs:
            cmd += ["--show-logs"]

    print("Running:", " ".join(cmd))
    print("Log ->", log_path)

    with open(log_path, "w", encoding="utf-8") as f:
        proc = subprocess.Popen(
            cmd,
            cwd=str(harness_dir),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        try:
            for line in proc.stdout:
                print(line, end="")
                f.write(line)
            rc = proc.wait(timeout=timeout_sec)
        except subprocess.TimeoutExpired:
            proc.kill()
            raise TimeoutError(f"Scoring timed out after {timeout_sec}s. See log: {log_path}")

    if rc != 0:
        raise RuntimeError(f"Scoring failed (exit={rc}). See log: {log_path}")

    print("
✅ FULL SCORING finished successfully.")
    return log_path

run_full_scoring(show_logs=True, mode="full", timeout_sec=None)


## I. Write score_status.json


In [ ]:
import json
import shutil
from pathlib import Path

results_dir = Path("/content/results/badblueprint")
results_dir.mkdir(parents=True, exist_ok=True)
log_path = results_dir / "full_score.log"
status_path = results_dir / "score_status.json"

ok = log_path.exists() and log_path.stat().st_size > 0

# Copy agent-card*.json into results_dir (optional, best-effort)
candidates = list(Path("/content/agentbeats-lambda").rglob("agent-card*.json")) + list(Path("/content").rglob("agent-card*.json"))
seen = set()
for src in candidates:
    name = src.name
    if name in seen:
        continue
    try:
        shutil.copy2(src, results_dir / name)
        seen.add(name)
    except Exception:
        pass

payload = {
    "ok": bool(ok),
    "results_dir": str(results_dir),
    "log_path": str(log_path),
}
status_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote:", status_path)


## J. Package results for download


In [ ]:
import tarfile
from pathlib import Path

results_dir = Path("/content/results/badblueprint")
archive_path = Path("/content/results_badblueprint_colab.tgz")

with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(results_dir, arcname="badblueprint")

print("Archive created at:", archive_path)


## Cleanup (stop server)


In [ ]:
import os
import signal
import time
from pathlib import Path

SERVER_PID = Path("/content/transformers_server.pid")
if SERVER_PID.exists():
    pid = int(SERVER_PID.read_text().strip())
    try:
        os.killpg(pid, signal.SIGTERM)
        time.sleep(2)
        try:
            os.kill(pid, 0)
            os.killpg(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        print(f"Stopped server process group {pid}")
    except ProcessLookupError:
        print("Server process not running.")
    SERVER_PID.unlink(missing_ok=True)
else:
    print("No server PID file found.")

